# S7_05: 건축공학 Claude Code 워크플로

**Building with the Claude API — Section 7, 도메인 특화 실습**

이 노트북에서는 건축공학 프로젝트에서 Claude Code를 활용하는 실전 워크플로를 실습합니다.

---

## 학습 목표
- 건축공학 도메인에 적합한 CLAUDE.md를 작성할 수 있다
- CPI 패턴으로 구조 검토 시스템을 단계적으로 구축할 수 있다
- KDS 기준 기반 계산 모듈과 MCP 서버를 설계할 수 있다

## 시나리오: RC 보 설계 검토 시스템 구축

Claude Code를 사용하여 KDS 14 20 20 기준 RC 보 설계 검토 시스템을 구축합니다.

### 검토 항목
1. 최소/최대 철근비 검토
2. 공칭 휨 강도 (Mn) 계산
3. 설계 휨 강도 (φMn) vs 소요 강도 (Mu) 비교
4. 공칭 전단 강도 (Vn) 계산
5. 설계 전단 강도 (φVn) vs 소요 전단력 (Vu) 비교

### 입력 변수
| 변수 | 설명 | 단위 |
| --- | --- | --- |
| b | 보 폭 | mm |
| d | 유효 깊이 | mm |
| fck | 콘크리트 설계기준강도 | MPa |
| fy | 철근 항복강도 | MPa |
| As | 인장 철근 면적 | mm² |
| Av | 전단 철근 면적 | mm² |
| s | 전단 철근 간격 | mm |
| Mu | 소요 휨 모멘트 | kN·m |
| Vu | 소요 전단력 | kN |

## v1: CLAUDE.md 작성

### 과제
RC 보 설계 검토 시스템을 위한 CLAUDE.md를 작성하세요.

In [ ]:
# v1: 건축공학 프로젝트 CLAUDE.md 작성
structural_claude_md = """
# CLAUDE.md

## 프로젝트 개요
KDS 14 20 20 기준 RC 보 설계 검토 자동화 시스템.
사용자가 부재 정보를 입력하면 휨/전단 검토 결과를 자동으로 산출한다.

## 기술 스택
- Python 3.11+
- FastAPI (백엔드 API)
- Streamlit (프론트엔드 UI)
- pytest (테스트)

## 디렉토리 구조
src/
  calculators/    구조 계산 모듈 (휨, 전단)
  api/            FastAPI 라우터
  ui/             Streamlit 페이지
  models/         Pydantic 데이터 모델
tests/            pytest 테스트

## 도메인 규칙
### 기준 코드
- KDS 14 20 20: 콘크리트 구조 설계 기준
- 변수명은 기준코드 표기법을 따른다 (fck, fy, As, Av, Mu, Vu)

### 단위 시스템
- 힘: kN, 모멘트: kN·m, 응력: MPa
- 길이: mm (단면 계산), m (부재 길이)
- 면적: mm² (철근, 단면)

### 안전 제약
- 강도감소계수 φ 반드시 적용 (휨: 0.85, 전단: 0.75)
- 최소 철근비 항상 검사
- 계산 결과에 단위를 명시

## 코딩 컨벤션
- 타입 힌트 필수
- Docstring: Google 스타일, 한국어 설명
- 함수명: snake_case
- 테스트: 각 계산 함수에 대한 단위 테스트 필수
"""

print(structural_claude_md)

## v2: CPI 패턴으로 모듈 구현

Claude Code에게 단계적으로 지시하는 프롬프트를 설계합니다.

In [ ]:
# v2: CPI 패턴 프롬프트 설계

# Context 프롬프트
print("=== Context 단계 ===")
print("""
CLAUDE.md를 읽고, src/calculators/ 디렉토리의 기존 코드를 확인해줘.
현재 어떤 계산 함수가 구현되어 있고, 어떤 패턴을 따르는지 분석해줘.
""")

# Plan 프롬프트
print("=== Plan 단계 ===")
print("""
KDS 14 20 20 기준으로 RC 보의 전단 강도 검토 모듈을 추가하려고 해.
다음 항목을 포함하는 구현 계획을 세워줘. 아직 코드를 작성하지 마.

1. 콘크리트 전단 강도 Vc 계산
2. 전단 철근 기여 강도 Vs 계산  
3. 공칭 전단 강도 Vn = Vc + Vs
4. 설계 전단 강도 φVn vs Vu 비교
5. 최소 전단 철근 검토
""")

# Implement 프롬프트
print("=== Implement 단계 ===")
print("""
계획대로 구현해줘. 다음을 포함해줘:
- src/calculators/shear_design.py 생성
- src/models/shear.py 에 입출력 Pydantic 모델 정의
- tests/test_shear_design.py 생성 및 실행
- 테스트 데이터: b=400, d=550, fck=27, fy=400, Av=142(D10@2), s=200, Vu=250kN
""")

## v2 참고: 전단 강도 계산 수식

### KDS 14 20 20 기준

**콘크리트 전단 강도:**
$$V_c = \frac{1}{6} \sqrt{f_{ck}} \cdot b_w \cdot d$$

**전단 철근 기여 강도:**
$$V_s = \frac{A_v \cdot f_y \cdot d}{s}$$

**공칭 전단 강도:**
$$V_n = V_c + V_s$$

**설계 전단 강도:**
$$\phi V_n \geq V_u \quad (\phi = 0.75)$$

In [ ]:
import math

def check_shear_design(
    b: float,      # 보 폭 (mm)
    d: float,      # 유효 깊이 (mm)
    fck: float,    # 콘크리트 설계기준강도 (MPa)
    fy: float,     # 철근 항복강도 (MPa)
    Av: float,     # 전단 철근 면적 (mm²)
    s: float,      # 전단 철근 간격 (mm)
    Vu: float      # 소요 전단력 (kN)
) -> dict:
    """KDS 14 20 20 기준 RC 보 전단 강도 검토.
    
    Args:
        b: 보 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
        Av: 전단 철근 면적 (mm²)
        s: 전단 철근 간격 (mm)
        Vu: 소요 전단력 (kN)
    
    Returns:
        검토 결과 딕셔너리
    """
    phi = 0.75  # 전단 강도감소계수
    
    # 콘크리트 전단 강도 (N)
    Vc = (1/6) * math.sqrt(fck) * b * d  # N
    
    # 전단 철근 기여 강도 (N)
    Vs = Av * fy * d / s  # N
    
    # 공칭 전단 강도 (N → kN)
    Vn = (Vc + Vs) / 1000  # kN
    
    # 설계 전단 강도
    phi_Vn = phi * Vn  # kN
    
    # 판정
    result = "OK" if phi_Vn >= Vu else "NG"
    ratio = phi_Vn / Vu if Vu > 0 else float('inf')
    
    return {
        "Vc_kN": round(Vc / 1000, 2),
        "Vs_kN": round(Vs / 1000, 2),
        "Vn_kN": round(Vn, 2),
        "phi_Vn_kN": round(phi_Vn, 2),
        "Vu_kN": Vu,
        "ratio": round(ratio, 3),
        "result": result
    }


# 테스트 실행
result = check_shear_design(
    b=400, d=550, fck=27, fy=400,
    Av=142, s=200, Vu=250
)

print("=== RC 보 전단 강도 검토 결과 ===")
for key, value in result.items():
    print(f"  {key}: {value}")

## v3: MCP 서버 연동 설계

### 과제
위 전단 강도 검토 기능을 MCP 서버로 감싸서 Claude Code에서 사용할 수 있도록 설계하세요.

In [ ]:
# v3: MCP 서버 설계 (FastMCP 기반)
mcp_server_design = """
# structural_calc_server.py (MCP 서버)

from fastmcp import FastMCP

mcp = FastMCP("structural-calc")

@mcp.tool()
def check_shear_strength(
    b: float,
    d: float,
    fck: float,
    fy: float,
    Av: float,
    s: float,
    Vu: float
) -> dict:
    \"\"\"KDS 14 20 20 기준 RC 보 전단 강도 검토\"\"\"
    # ... (위의 check_shear_design 함수와 동일)
    pass

@mcp.tool()
def check_flexural_strength(
    b: float,
    d: float,
    fck: float,
    fy: float,
    As: float,
    Mu: float
) -> dict:
    \"\"\"KDS 14 20 20 기준 RC 보 휨 강도 검토\"\"\"
    pass

@mcp.tool()
def get_kds_requirement(section_code: str, item: str) -> str:
    \"\"\"KDS 기준 조항 조회\"\"\"
    pass
"""

print(mcp_server_design)
print()
print("=== Claude Code 설정 ===")
print("""
# .claude/settings.json에 추가:
{
  "mcpServers": {
    "structural-calc": {
      "command": "python",
      "args": ["-m", "structural_calc_server"],
      "cwd": "/path/to/server"
    }
  }
}

# Claude Code에서 사용:
# > fck=27MPa, fy=400MPa, b=400mm, d=550mm일 때
#   전단 철근 D10@200으로 Vu=250kN에 대한 전단 검토를 해줘.
""")

## 도전 과제

1. **CLAUDE.md 확장**: 위 CLAUDE.md에 "자주 하는 실수" 섹션을 추가하세요
   - 예: 강도감소계수 누락, 단위 불일치, 최소 철근비 미검토 등

2. **CPI 실전**: 실제 Claude Code에서 위 CPI 프롬프트를 실행하고 결과를 비교하세요

3. **MCP 구현**: FastMCP로 위 MCP 서버를 실제로 구현하고 Claude Code에 연동하세요

---

## 참고 자료
- 강의노트: Week_08.md §2.6 (건축공학 활용 시나리오)
- KDS 14 20 20: 콘크리트 구조 설계 기준
- [FastMCP 문서](https://github.com/jlowin/fastmcp)